"""
Transpilación para hardware IBM
===================================================
Fake backend FakeSherbrooke: réplica de un Eagle de 127 qubits.
Sin cuenta IBM, sin Runtime, 100% local.

In [1]:
!pip install qiskit qiskit-ibm-runtime

In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# ── 1. Backend: Eagle 127q (local, sin cuenta IBM) ────────────────────────────
# Basis gates reales: ['ecr', 'id', 'rz', 'sx', 'x']
# Topología: heavy-hex lattice
backend = FakeSherbrooke()

In [3]:
# ── 2. Circuito original (puertas abstractas) ─────────────────────────────────
qc = QuantumCircuit(3, 3)
qc.h(0)        # Hadamard   → no nativa en Eagle
qc.cx(0, 1)    # CNOT       → no nativa en Eagle
qc.cx(1, 2)    # CNOT
qc.t(0)        # T gate     → no nativa en Eagle
qc.s(1)        # S gate     → no nativa en Eagle
qc.measure([0, 1, 2], [0, 1, 2])

print("CIRCUITO ORIGINAL")
print("=================")
print(qc.draw(output="text"))
print(f"Puertas: {dict(qc.count_ops())}")
print(f"Profundidad: {qc.depth()}")

CIRCUITO ORIGINAL
     ┌───┐     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤ T ├─────┤M├───
     └───┘┌─┴─┐└───┘┌───┐└╥┘┌─┐
q_1: ─────┤ X ├──■──┤ S ├─╫─┤M├
          └───┘┌─┴─┐└┬─┬┘ ║ └╥┘
q_2: ──────────┤ X ├─┤M├──╫──╫─
               └───┘ └╥┘  ║  ║ 
c: 3/═════════════════╩═══╩══╩═
                      2   0  1 
Puertas: {'measure': 3, 'cx': 2, 'h': 1, 't': 1, 's': 1}
Profundidad: 5


In [4]:
# ── 3. Transpilación ──────────────────────────────────────────────────────────
# El transpiler:
#   • Descompone H, CX, T, S → ECR + RZ + SX + X (basis gates de Eagle)
#   • Asigna qubits lógicos a qubits físicos del chip (layout)
#   • Inserta SWAPs si el circuito pide CNOTs entre qubits no conectados
t_qc = transpile(
    qc,
    backend=backend,
    optimization_level=1,   # 0=sin opt · 1=ligera · 2=pesada · 3=máxima
    seed_transpiler=42,     # resultado reproducible
)

In [5]:
print("\nCIRCUITO TRANSPILADO (solo puertas nativas IBM Eagle)")
print("======================================================")
print(t_qc.draw(output="text"))
print(f"Puertas: {dict(t_qc.count_ops())}")
print(f"Profundidad: {t_qc.depth()}")


CIRCUITO TRANSPILADO (solo puertas nativas IBM Eagle)
global phase: 5π/8
          ┌────────┐ ┌────┐          ┌──────┐┌──────────┐┌────┐┌──────────┐»
q_0 -> 0 ─┤ Rz(-π) ├─┤ √X ├──────────┤1     ├┤ Rz(-π/2) ├┤ √X ├┤ Rz(3π/4) ├»
         ┌┴────────┴┐├────┤┌────────┐│  Ecr │├─────────┬┘├────┤└─┬──────┬─┘»
q_1 -> 1 ┤ Rz(-π/2) ├┤ √X ├┤ Rz(-π) ├┤0     ├┤ Rz(π/2) ├─┤ √X ├──┤0     ├──»
         └┬────────┬┘├────┤├────────┤└──────┘└─────────┘ └────┘  │  Ecr │  »
q_2 -> 2 ─┤ Rz(-π) ├─┤ √X ├┤ Rz(-π) ├────────────────────────────┤1     ├──»
          └────────┘ └────┘└────────┘                            └──────┘  »
    c: 3/══════════════════════════════════════════════════════════════════»
                                                                           »
«              ┌─┐              
«q_0 -> 0 ─────┤M├──────────────
«         ┌───┐└╥┘┌─────────┐┌─┐
«q_1 -> 1 ┤ X ├─╫─┤ Rz(π/2) ├┤M├
«         └┬─┬┘ ║ └─────────┘└╥┘
«q_2 -> 2 ─┤M├──╫─────────────╫─
«          └╥┘  ║             ║ 
« 